# Kentucky From Above — Using AbovePy + Occulus

This notebook shows how to use **AbovePy** (by Chris Lyons) to download LiDAR
tiles from the [Kentucky From Above](https://kyfromabove.ky.gov/) program
and then analyse them with **Occulus**.

## Installation
```bash
pip install abovepy[lidar]
pip install occulus[all]
```

KY From Above provides free statewide LiDAR data at 1–2 ft nominal pulse spacing
across all 120 Kentucky counties.

In [ ]:
# Check both packages are installed
import abovepy
import occulus
print('abovepy :', abovepy.__version__)
print('occulus :', occulus.__version__)

## 1. Search for Available LiDAR Tiles

AbovePy can search by county name or bounding box.

In [ ]:
from abovepy import search, download

# Search for Phase 3 LiDAR tiles in Fayette County (Lexington)
tiles = search('lidar', county='fayette', phase=3)
print(f'Found {len(tiles)} tiles for Fayette County (Phase 3)')
for t in tiles[:3]:
    print(f'  {t}')

## 2. Download a Tile

In [ ]:
import tempfile
from pathlib import Path

download_dir = Path(tempfile.mkdtemp()) / 'ky_lidar'
download_dir.mkdir(parents=True, exist_ok=True)

# Download the first tile
if tiles:
    laz_path = download(tiles[0], dest=download_dir)
    print(f'Downloaded: {laz_path}')
    print(f'File size : {laz_path.stat().st_size / 1e6:.1f} MB')
else:
    print('No tiles found — check county name or phase.')

## 3. Load into Occulus

In [ ]:
from occulus.io import read
from occulus.metrics import compute_cloud_statistics

cloud = read(laz_path, platform='aerial', subsample=0.25)
print(cloud)

stats = compute_cloud_statistics(cloud)
print(f'\nElevation range : {stats.z_min:.2f} – {stats.z_max:.2f} m (NAVD 88)')
print(f'Point density   : ~{cloud.n_points / (cloud.bounds[1,0]-cloud.bounds[0,0]) / (cloud.bounds[1,1]-cloud.bounds[0,1]):.1f} pts/m²')

## 4. Ground Classification

In [ ]:
import numpy as np
from occulus.segmentation import classify_ground_csf
from occulus.types import AerialCloud

# Use pre-classified labels if present (KY From Above tiles are pre-classified)
if isinstance(cloud, AerialCloud) and cloud.classification is not None:
    n_ground = int((cloud.classification == 2).sum())
    print(f'Pre-classified ground: {n_ground:,} ({n_ground/cloud.n_points:.1%})')
    classified = cloud
else:
    print('Running CSF ground classification…')
    classified = classify_ground_csf(cloud)
    if isinstance(classified, AerialCloud) and classified.classification is not None:
        n_ground = int((classified.classification == 2).sum())
        print(f'CSF ground: {n_ground:,} ({n_ground/cloud.n_points:.1%})')

## 5. Canopy Height Model + Visualisation

In [ ]:
import matplotlib.pyplot as plt
from occulus.metrics import canopy_height_model, coverage_statistics

if isinstance(classified, AerialCloud):
    chm, xe, ye = canopy_height_model(classified, resolution=0.5)
    cov = coverage_statistics(classified, resolution=0.5)

    valid = chm[~np.isnan(chm)]
    print(f'Max canopy height : {valid.max():.1f} m')
    print(f'Gap fraction      : {cov.gap_fraction:.2%}')
    print(f'Mean density      : {cov.mean_density:.1f} pts/m²')

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(chm, origin='lower', cmap='Greens',
                   extent=[xe[0], xe[-1], ye[0], ye[-1]])
    plt.colorbar(im, ax=ax, label='Height above ground (m)')
    ax.set_title(f'Canopy Height Model — Fayette County, KY')
    ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
    plt.tight_layout()
    plt.savefig('../../outputs/kyfromabove_fayette_chm.png', dpi=150)
    plt.show()

## 6. Stream Data (No Download)

AbovePy can stream point data directly without writing to disk,
returning a pandas DataFrame that you can wrap into Occulus.

In [ ]:
# Stream a small geographic window
# bbox = [min_lon, min_lat, max_lon, max_lat]  (WGS84)
# This covers a small area of downtown Lexington, KY
bbox_wgs84 = [-84.503, 38.045, -84.495, 38.052]

try:
    from abovepy import read as ky_read
    df = ky_read('lidar', bbox=bbox_wgs84, phase=3)
    print(f'Streamed {len(df):,} points')
    print(df[['x', 'y', 'z']].describe())

    # Wrap in Occulus
    from occulus.types import AerialCloud
    import numpy as np
    stream_cloud = AerialCloud(df[['x', 'y', 'z']].values.astype(np.float64))
    print(stream_cloud)
except Exception as exc:
    print(f'Stream read failed: {exc}')

---
For more on AbovePy: `pip show abovepy` or visit https://pypi.org/project/abovepy/

**Next**: `03_ground_classification.ipynb`